# Feature engineering

In [147]:
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np 
import seaborn as sns

In [148]:
df = pd.read_excel('/Users/baptistesmac/Documents/Formation finance/ML ligue 1/data/raw/understat2_ligue1_matches.xlsx')

## 1. Aggregate points before a match

Let's add a relevant variable : compute the number of point of each team before a match. 

In [149]:
def home_pts(result):
    if result == 'H': return 3
    if result == 'D': return 1
    return 0

def away_pts(result):
    if result == 'A': return 3
    if result == 'D': return 1
    return 0

df['_home_pts'] = df['Result'].apply(home_pts)
df['_away_pts'] = df['Result'].apply(away_pts)

home_view = df[['season', 'date', 'home_team', '_home_pts']].copy()
home_view.columns = ['season', 'date', 'team', 'pts']

away_view = df[['season', 'date', 'away_team', '_away_pts']].copy()
away_view.columns = ['season', 'date', 'team', 'pts']

all_matches = pd.concat([home_view, away_view]).sort_values(['season', 'team', 'date'])

all_matches['cumul_pts'] = all_matches.groupby(
    ['season', 'team']
)['pts'].transform(lambda x: x.cumsum().shift(1).fillna(0))

home_pts_map = all_matches.set_index(['season', 'date', 'team'])['cumul_pts']

df['home_points'] = df.apply(
    lambda row: home_pts_map.get((row['season'], row['date'], row['home_team']), 0),
    axis=1
)
df['away_points'] = df.apply(
    lambda row: home_pts_map.get((row['season'], row['date'], row['away_team']), 0),
    axis=1
)

df = df.drop(columns=['_home_pts', '_away_pts'])

df[df['season'] == '2020/21'][['date', 'home_team', 'away_team', 'Result', 'home_points', 'away_points']].head(10)

,date,home_team,away_team,Result,home_points,away_points
1418,2020-08-21,Bordeaux,Nantes,D,0.0,0.0
1419,2020-08-22,Dijon,Angers,A,0.0,0.0
1420,2020-08-22,Lille,Rennes,D,0.0,0.0
1421,2020-08-23,Monaco,Reims,D,0.0,0.0
1422,2020-08-23,Nimes,Brest,H,0.0,0.0
1423,2020-08-23,Lorient,Strasbourg,H,0.0,0.0
1424,2020-08-23,Nice,Lens,H,0.0,0.0
1425,2020-08-28,Lyon,Dijon,H,0.0,0.0
1426,2020-08-29,Strasbourg,Nice,A,0.0,3.0
1427,2020-08-29,Rennes,Montpellier,H,1.0,0.0


## 2. Rolling features

Computing the rolling feature of scored and conceded goals for each team, xG, % of wins etc.


In [150]:
def add_rolling_feature(df, home_col, away_col, feature_name, window=5):
    #Computes the rolling of a feature for each team, and adds it to df 
    
    home_view = df[["date", "season", "home_team", home_col, away_col]].copy()
    home_view.columns = ["date", "season", "team", "created", "conceded"]
    
    away_view = df[["date", "season", "away_team", away_col, home_col]].copy()
    away_view.columns = ["date", "season", "team", "created", "conceded"]
    
    all_matches = pd.concat([home_view, away_view]).sort_values(["season", "team", "date"])
    
    all_matches[f'rolling_{feature_name}_created'] = all_matches.groupby(
        ['season', 'team']
    )['created'].transform(lambda x: x.rolling(window).mean().shift(1))
    
    all_matches[f'rolling_{feature_name}_conceded'] = all_matches.groupby(
        ['season', 'team']
    )['conceded'].transform(lambda x: x.rolling(window).mean().shift(1))
    
    feat_map_created  = all_matches.set_index(['season', 'date', 'team'])[f'rolling_{feature_name}_created']
    feat_map_conceded = all_matches.set_index(['season', 'date', 'team'])[f'rolling_{feature_name}_conceded']
    
    df[f'home_rolling_{feature_name}_created'] = df.apply(
        lambda row: feat_map_created.get((row['season'], row['date'], row['home_team']), 0), axis=1
    )
    df[f'away_rolling_{feature_name}_created'] = df.apply(
        lambda row: feat_map_created.get((row['season'], row['date'], row['away_team']), 0), axis=1
    )
    df[f'home_rolling_{feature_name}_conceded'] = df.apply(
        lambda row: feat_map_conceded.get((row['season'], row['date'], row['home_team']), 0), axis=1
    )
    df[f'away_rolling_{feature_name}_conceded'] = df.apply(
        lambda row: feat_map_conceded.get((row['season'], row['date'], row['away_team']), 0), axis=1
    )
    
    return df

Now, let's do the same for the rolling % of wins : 

In [151]:
def add_rolling_winrate(df, window=5):
    
    # For home  win = "H"
    home_view = df[["date", "season", "home_team", "Result"]].copy()
    home_view.columns = ["date", "season", "team", "result"]
    home_view["win"] = (home_view["result"] == "H").astype(int)

    # For away : win = "A"
    away_view = df[["date", "season", "away_team", "Result"]].copy()
    away_view.columns = ["date", "season", "team", "result"]
    away_view["win"] = (away_view["result"] == "A").astype(int)

    all_matches = pd.concat([home_view, away_view]).sort_values(["season", "team", "date"])

    all_matches["rolling_winrate"] = all_matches.groupby(
        ["season", "team"]
    )["win"].transform(lambda x: x.rolling(window).mean().shift(1))

    win_map = all_matches.set_index(["season", "date", "team"])["rolling_winrate"]

    df["home_rolling_winrate"] = df.apply(
        lambda row: win_map.get((row["season"], row["date"], row["home_team"]), 0), axis=1
    )
    df["away_rolling_winrate"] = df.apply(
        lambda row: win_map.get((row["season"], row["date"], row["away_team"]), 0), axis=1
    )

    return df

In [152]:
df = add_rolling_feature(df, 'home_goals', 'away_goals', 'goals')
df = add_rolling_feature(df, 'home_xg',    'away_xg',    'xg')
df = add_rolling_feature(df, 'home_ppda',  'away_ppda',  'ppda')
df = add_rolling_winrate(df)

Same, but we separete xG when playing home and away : 

In [153]:
def add_rolling_feature_home_away(df, home_col, away_col, feature_name, window=5):
    #separate home and away games

    home_view = df[["date", "season", "home_team", home_col]].copy()
    home_view.columns = ["date", "season", "team", "value"]

    home_view["rolling"] = home_view.groupby(
        ["season", "team"]
    )["value"].transform(lambda x: x.rolling(window).mean().shift(1))

    home_map = home_view.set_index(["season", "date", "team"])["rolling"]

    away_view = df[["date", "season", "away_team", away_col]].copy()
    away_view.columns = ["date", "season", "team", "value"]

    away_view["rolling"] = away_view.groupby(
        ["season", "team"]
    )["value"].transform(lambda x: x.rolling(window).mean().shift(1))

    away_map = away_view.set_index(["season", "date", "team"])["rolling"]

    df[f"home_rolling_{feature_name}_home_only"] = df.apply(
        lambda row: home_map.get((row["season"], row["date"], row["home_team"]), 0), axis=1
    )

    df[f"away_rolling_{feature_name}_away_only"] = df.apply(
        lambda row: away_map.get((row["season"], row["date"], row["away_team"]), 0), axis=1
    )

    return df

In [154]:
df = add_rolling_feature_home_away(df, 'home_xg', 'away_xg', 'xg')
df = add_rolling_feature_home_away(df, 'home_goals', 'away_goals', 'goals')

### Feature : differentials between two teams

$\rightarrow$ of points, xG rolling, PPDA, goals, winrate

In [155]:
df["points_diff"] = df["home_points"]-df["away_points"]
df["rolling_xG_diff"] = df["home_rolling_xg_created"]-df["away_rolling_xg_created"]
df["rolling_ppda_diff"]= df["home_rolling_ppda_created"]-df["away_rolling_ppda_created"]
df["rolling_goals_diff"] = df["home_rolling_goals_created"] - df["away_rolling_goals_created"]
df["rolling_winrate_diff"] = df["home_rolling_winrate"] - df["away_rolling_winrate"]

## 3. Adding teams market value as a variable

In [156]:
df_values = pd.read_excel('/Users/baptistesmac/Documents/Formation finance/ML ligue 1/data/raw/transfermarkt_values.xlsx')
df_values['season'] = df_values['season'].apply(
    lambda s: s[:4] + '/' + s[7:9]  # "2016/2017" → "2016/17"
)

df_values['value'] = df_values['value'].apply(parse_value)

df = df.merge(
    df_values.rename(columns={
        'team': 'home_team',
        'value': 'home_squad_value'
    }),
    on=['season', 'home_team'],
    how='left'
)

# Away
df = df.merge(
    df_values.rename(columns={
        'team': 'away_team',
        'value': 'away_squad_value'
    }),
    on=['season', 'away_team'],
    how='left'
)

# Features dérivées
df['squad_value_ratio'] = df['home_squad_value'] / df['away_squad_value']
df['squad_value_diff']  = df['home_squad_value'] - df['away_squad_value']

## 4. Targeted variable

Let's encode the result to numbers : H $\to$ 0, D $\to$ 1, A $\to$ 2

In [157]:
df["target"] = df["Result"].map({"H":0,"D":1,"A":2})

We save new features into a new csv : 

In [158]:
df.to_csv("/Users/baptistesmac/Documents/Formation finance/ML ligue 1/data/ligue1_features2.csv", index=False)
df.head()
df.isnull().sum()

season                            0
date                              0
home_team                         0
away_team                         0
home_goals                        0
away_goals                        0
home_xg                           0
away_xg                           0
home_shots                        0
away_shots                        0
home_shots_on_target              0
away_shots_on_target              0
home_ppda                         0
away_ppda                         0
home_deep                         0
away_deep                         0
Result                            0
home_points                       0
away_points                       0
home_rolling_goals_created      486
away_rolling_goals_created      484
home_rolling_goals_conceded     486
away_rolling_goals_conceded     484
home_rolling_xg_created         486
away_rolling_xg_created         484
home_rolling_xg_conceded        486
away_rolling_xg_conceded        484
home_rolling_ppda_created   